In [9]:
# ============================================================
# 01.1 — SETUP & TARGET IDENTIFICATION
# ============================================================

import os
import time
import requests
import pandas as pd

from datetime import datetime, timezone
from urllib.parse import urljoin

# -----------------------------
# PROJECT CONFIGURATION
# -----------------------------

BASE_DIR = "/content/drive/MyDrive/PROJECT SAINS DATA"
DATA_DIR = os.path.join(BASE_DIR, "DATA")
ALPHA_DIR = os.path.join(DATA_DIR, "Alpha-Synuclein")

os.makedirs(ALPHA_DIR, exist_ok=True)

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data/"

TARGET_ID = "CHEMBL6152"
EXPECTED_UNIPROT = "P37840"
LIMIT = 1000

session = requests.Session()
session.headers.update({"Accept": "application/json"})

# -----------------------------
# PAGINATION FUNCTION
# -----------------------------

def fetch_chembl_data(endpoint, result_key, target_id):
    url = urljoin(BASE_URL, f"{endpoint}.json")

    params = {
        "target_chembl_id": target_id,
        "limit": LIMIT,
        "offset": 0
    }

    records = []
    page_count = 0
    expected_total = None

    while url:
        response = session.get(
            url,
            params=params,
            timeout=60
        )
        response.raise_for_status()

        data = response.json()
        page_meta = data.get("page_meta", {})

        if expected_total is None:
            expected_total = page_meta.get("total_count")

        page_records = data.get(result_key, [])
        records.extend(page_records)
        page_count += 1

        print(
            f"Page {page_count}: "
            f"{len(page_records)} records | "
            f"Total: {len(records)}"
        )

        next_page = page_meta.get("next")
        url = urljoin(BASE_URL, next_page) if next_page else None

        # Parameter hanya digunakan pada request pertama
        params = None
        time.sleep(0.2)

    return pd.DataFrame(records), expected_total, page_count


# -----------------------------
# TARGET IDENTIFICATION
# -----------------------------

target_url = urljoin(
    BASE_URL,
    f"target/{TARGET_ID}.json"
)

response = session.get(target_url, timeout=60)
response.raise_for_status()

target_data = response.json()

components = target_data.get("target_components", [])

uniprot_accessions = [
    item.get("accession")
    for item in components
    if item.get("accession")
]

if EXPECTED_UNIPROT not in uniprot_accessions:
    raise ValueError(
        f"Target verification failed: {uniprot_accessions}"
    )

alpha_target_df = pd.DataFrame([{
    "Target_Name": target_data.get("pref_name"),
    "ChEMBL_Target_ID": TARGET_ID,
    "UniProt_Accession": EXPECTED_UNIPROT,
    "Organism": target_data.get("organism"),
    "Target_Type": target_data.get("target_type"),
    "Taxonomy_ID": target_data.get("tax_id"),
    "Gene": "SNCA",
    "Data_Source": "ChEMBL API",
    "Retrieval_Date": datetime.now(timezone.utc).isoformat()
}])

target_output = os.path.join(
    ALPHA_DIR,
    "alpha_synuclein_target_metadata.csv"
)

if not os.path.exists(target_output):
    alpha_target_df.to_csv(target_output, index=False)
else:
    print("Target metadata sudah tersedia. File lama dipertahankan.")

print("\nTARGET VERIFICATION: SUCCESS")
display(alpha_target_df)

Target metadata sudah tersedia. File lama dipertahankan.

TARGET VERIFICATION: SUCCESS


,Target_Name,ChEMBL_Target_ID,UniProt_Accession,Organism,Target_Type,Taxonomy_ID,Gene,Data_Source,Retrieval_Date
0,Alpha-synuclein,CHEMBL6152,P37840,Homo sapiens,SINGLE PROTEIN,9606,SNCA,ChEMBL API,2026-11-27T06:38:23.538503+00:00


In [10]:
# ============================================================
# 01.2 — ACTIVITY DATA COLLECTION
# ============================================================

ACTIVITY_OUTPUT = os.path.join(
    ALPHA_DIR,
    "alpha_synuclein_activity_raw.csv"
)

if os.path.exists(ACTIVITY_OUTPUT):

    print("Activity dataset sudah tersedia. Membaca file lama.")

    alpha_activity_raw = pd.read_csv(
        ACTIVITY_OUTPUT,
        low_memory=False
    )

else:

    print("Mengambil activity data dari ChEMBL API...\n")

    alpha_activity_raw, expected_total, page_count = (
        fetch_chembl_data(
            endpoint="activity",
            result_key="activities",
            target_id=TARGET_ID
        )
    )

    if alpha_activity_raw.empty:
        raise ValueError("Activity dataset kosong.")

    # Target consistency check
    retrieved_targets = set(
        alpha_activity_raw["target_chembl_id"].dropna()
    )

    if retrieved_targets != {TARGET_ID}:
        raise ValueError(
            f"Target tidak konsisten: {retrieved_targets}"
        )

    alpha_activity_raw.to_csv(
        ACTIVITY_OUTPUT,
        index=False
    )

    print("\nActivity dataset berhasil disimpan.")

# Final summary
print("\nACTIVITY COLLECTION SUMMARY")
print("Target ID       :", TARGET_ID)
print("Total records   :", len(alpha_activity_raw))
print("Total columns   :", len(alpha_activity_raw.columns))
print("Unique compounds:",
      alpha_activity_raw["molecule_chembl_id"].nunique())
print("Unique assays   :",
      alpha_activity_raw["assay_chembl_id"].nunique())

display(alpha_activity_raw.head())

Activity dataset sudah tersedia. Membaca file lama.

ACTIVITY COLLECTION SUMMARY
Target ID       : CHEMBL6152
Total records   : 12221
Total columns   : 47
Unique compounds: 11054
Unique assays   : 429


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,NaN,NaN,2085731,[],CHEMBL945801,Binding affinity to alpha-synuclein fibrils by...,B,NaN,NaN,BAO_0000587,...,Homo sapiens,Alpha-synuclein,9606,NaN,NaN,Kb,10^6/M,NaN,NaN,1.78
1,NaN,NaN,2085732,[],CHEMBL945801,Binding affinity to alpha-synuclein fibrils by...,B,NaN,NaN,BAO_0000587,...,Homo sapiens,Alpha-synuclein,9606,NaN,NaN,Kb,10^6/M,NaN,NaN,1.53
2,NaN,NaN,3396892,[],CHEMBL1218779,Inhibition of alpha-synuclein-mediated toxicit...,B,NaN,NaN,BAO_0000201,...,Homo sapiens,Alpha-synuclein,9606,Active,NaN,INH,NaN,UO_0000187,NaN,NaN
3,NaN,NaN,3396893,[],CHEMBL1218779,Inhibition of alpha-synuclein-mediated toxicit...,B,NaN,NaN,BAO_0000201,...,Homo sapiens,Alpha-synuclein,9606,Active,NaN,INH,NaN,UO_0000187,NaN,NaN
4,NaN,NaN,3396898,[],CHEMBL1218780,Inhibition of human alpha-synuclein-mediated a...,B,NaN,NaN,BAO_0000375,...,Homo sapiens,Alpha-synuclein,9606,Not Active,NaN,Activity,NaN,NaN,NaN,NaN


In [11]:
# ============================================================
# 01.3 — ASSAY METADATA COLLECTION
# ============================================================

ASSAY_OUTPUT = os.path.join(
    ALPHA_DIR,
    "alpha_synuclein_assay_metadata_raw.csv"
)

if os.path.exists(ASSAY_OUTPUT):

    print("Assay metadata sudah tersedia. Membaca file lama.")

    alpha_assay_raw = pd.read_csv(
        ASSAY_OUTPUT,
        low_memory=False
    )

else:

    print("Mengambil assay metadata dari ChEMBL API...\n")

    alpha_assay_raw, expected_assays, assay_pages = (
        fetch_chembl_data(
            endpoint="assay",
            result_key="assays",
            target_id=TARGET_ID
        )
    )

    if alpha_assay_raw.empty:
        raise ValueError("Assay metadata kosong.")

    retrieved_targets = set(
        alpha_assay_raw["target_chembl_id"].dropna()
    )

    if retrieved_targets != {TARGET_ID}:
        raise ValueError(
            f"Target tidak konsisten: {retrieved_targets}"
        )

    alpha_assay_raw.to_csv(
        ASSAY_OUTPUT,
        index=False
    )

    print("\nAssay metadata berhasil disimpan.")

# -----------------------------
# ASSAY ID CROSS-CHECK
# -----------------------------

activity_assay_ids = set(
    alpha_activity_raw["assay_chembl_id"].dropna()
)

metadata_assay_ids = set(
    alpha_assay_raw["assay_chembl_id"].dropna()
)

matched_assays = activity_assay_ids & metadata_assay_ids
missing_assays = activity_assay_ids - metadata_assay_ids

print("\nASSAY COLLECTION SUMMARY")
print("Activity assay IDs :", len(activity_assay_ids))
print("Metadata assay IDs :", len(metadata_assay_ids))
print("Matched assay IDs  :", len(matched_assays))
print("Missing assay IDs  :", len(missing_assays))

display(alpha_assay_raw.head())

Assay metadata sudah tersedia. Membaca file lama.

ASSAY COLLECTION SUMMARY
Activity assay IDs : 429
Metadata assay IDs : 429
Matched assay IDs  : 429
Missing assay IDs  : 0


,aidx,assay_category,assay_cell_type,assay_chembl_id,assay_classifications,assay_group,assay_organism,assay_parameters,assay_strain,assay_subcellular_fraction,...,confidence_score,description,document_chembl_id,relationship_description,relationship_type,src_assay_id,src_id,target_chembl_id,tissue_chembl_id,variant_sequence
0,CLD0,NaN,NaN,CHEMBL945801,[],NaN,NaN,[],NaN,NaN,...,8,Binding affinity to alpha-synuclein fibrils by...,CHEMBL1143231,Homologous protein target assigned,H,NaN,1,CHEMBL6152,NaN,NaN
1,CLD0,NaN,NaN,CHEMBL1218779,[],NaN,Homo sapiens,[],NaN,NaN,...,9,Inhibition of alpha-synuclein-mediated toxicit...,CHEMBL1212724,Direct protein target assigned,D,NaN,1,CHEMBL6152,NaN,NaN
2,CLD0,NaN,NaN,CHEMBL1218780,[],NaN,Homo sapiens,[],NaN,NaN,...,9,Inhibition of human alpha-synuclein-mediated a...,CHEMBL1212724,Direct protein target assigned,D,NaN,1,CHEMBL6152,NaN,NaN
3,CLD0,NaN,NaN,CHEMBL1218781,[],NaN,Homo sapiens,[],NaN,NaN,...,9,Inhibition of alpha-synuclein-mediated dopamin...,CHEMBL1212724,Direct protein target assigned,D,NaN,1,CHEMBL6152,NaN,NaN
4,CLD0,NaN,NaN,CHEMBL1250260,[],NaN,NaN,[],NaN,NaN,...,8,Induction of alpha-synuclein aggregation in PB...,CHEMBL1240308,Homologous protein target assigned,H,NaN,1,CHEMBL6152,NaN,NaN


In [12]:
# Save collection log
log_path = os.path.join(
    ALPHA_DIR,
    "alpha_synuclein_collection_log.csv"
)

collection_log.to_csv(log_path, index=False)

print("=" * 65)
print("ALPHA-SYNUCLEIN COLLECTION LOG")
print("=" * 65)

display(collection_log)

print("\nCollection log saved successfully:")
print(log_path)

ALPHA-SYNUCLEIN COLLECTION LOG


,Dataset,Source,API_Endpoint,Target_ID,Total_Records,Total_Columns,Output_File,Retrieval_Date_UTC
0,Target Metadata,ChEMBL API,target/CHEMBL6152.json,CHEMBL6152,1,9,alpha_synuclein_target_metadata.csv,2026-11-27T06:32:59.816122+00:00
1,Activity Raw,ChEMBL API,activity.json,CHEMBL6152,12221,47,alpha_synuclein_activity_raw.csv,2026-11-27T06:32:59.816122+00:00
2,Assay Metadata,ChEMBL API,assay.json,CHEMBL6152,429,29,alpha_synuclein_assay_metadata_raw.csv,2026-11-27T06:32:59.816122+00:00



Collection log saved successfully:
/content/drive/MyDrive/PROJECT SAINS DATA\DATA\Alpha-Synuclein\alpha_synuclein_collection_log.csv
